In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pickle 

In [2]:
class LayerDense:

    def __init__(self,n_input,n_neuron):
        self.weights = np.random.randn(n_input, n_neuron) * np.sqrt(2 / n_input)
        self.biases = np.zeros((1,n_neuron))

        self.weight_momentumn = np.zeros_like(self.weights)
        self.biases_momentumn = np.zeros_like(self.biases)

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs,self.weights) + self.biases

    def backward(self, dvalues):
        self.d_biases = np.sum(dvalues,axis=0,keepdims=True)
        self.d_weights = np.dot(self.inputs.T,dvalues)
        self.d_inputs = np.dot(dvalues,self.weights.T)


class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0,inputs)

    def backward(self,dvalues):
        self.d_inputs = np.where(self.inputs <= 0, 0, dvalues)

class tanh:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = (np.exp(inputs) - np.exp(-inputs)) / (np.exp(inputs) + np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (1-self.output**2)


class sigmoid:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = 1/ (1+np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (self.output) * (1-self.output)


class softmax:
    def forward(self,inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs,axis=1,keepdims=True))
        self.output = exp_values / np.sum(exp_values,axis = 1, keepdims= True)

    def backward(self,dvalues):
        diagonal = np.diag(self.output)[:None]
        pad = np.zeros(diagonal.shape)
        sii = np.hstack((np.diagflat(diagonal),pad))
        self.d_inputs = sii - (diagonal * self.output)
        #in progress, let skip it as it's not needed


class Loss:
    def forward(self,y_pred,y_true):
        self.batch_size = y_pred.shape[0]
        self.y_pred = y_pred
        self.y_true = y_true
        p = y_pred[y_true == 1]
        p = np.clip(p,0+1e-9,1 - 1e-9)
        self.loss =  - np.average(np.log(p))
    
    def backward(self):
        y_pred = np.clip(self.y_pred,0+1e-9,1 - 1e-9)
        self.d_loss =  - self.y_true/ (y_pred*self.batch_size)

    def backward_with_softmax(self):
        self.d_inputs = self.y_pred - self.y_true


class optimizer:
    def __init__(self,lr = 0.001, beta = 0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self,layer):
        layer.weights -= self.lr*layer.d_weights
        layer.biases -= self.lr*layer.d_biases

    def OptimizerGD_WithMomentumn(self, layer):
        
        layer.weight_momentumn = self.beta * layer.weight_momentumn - self.lr*layer.d_weights
        layer.biases_momentumn = self.beta * layer.biases_momentumn - self.lr*layer.d_biases

        layer.weights += layer.weight_momentumn
        layer.biases += layer.biases_momentumn

In [3]:
class Rnn:

    def __init__(self,n_vocab,hidden_size):
        self.hidden_size = hidden_size
        self.n_vocab = n_vocab
        self.w_x = np.random.randn(n_vocab, hidden_size) * 0.01
        self.w_h = np.random.randn(hidden_size, hidden_size) * 0.01
        self.w_y = np.random.randn( hidden_size, n_vocab) * 0.01

        self.b_h = np.zeros((1,hidden_size))
        self.b_y = np.zeros((1,n_vocab))


        self.w_x_moment = np.zeros_like(self.w_x)
        self.w_h_moment = np.zeros_like(self.w_h)
        self.w_y_moment = np.zeros_like(self.w_y)

        self.b_y_moment = np.zeros_like(self.b_y)
        self.b_h_moment = np.zeros_like(self.b_h)


    def info_squeezing(self,hidden, activation):
        if activation == 'tanh':
            act = tanh()
            act.forward(hidden)

        elif activation == 'sigmoid':
            act = sigmoid()
            act.forward(hidden)

        else:
            raise ValueError('only tanh or sigmoid')

        return act.output
        

    def info_unsqueeze(self, dvalues, output, activation):
        if activation == 'tanh':
            act = tanh()
            act.output = output
            act.backward(dvalues)

        elif activation == 'sigmoid':
            act = sigmoid()
            act.output = output
            act.backward(dvalues)
        
        else:
            raise ValueError('only tanh or sigmoid')

        return act.d_inputs
        

    def cell_forward(self, h_previous, h_current, activation):
        hidden = np.dot(h_current,self.w_x) + self.b_h + h_previous @ self.w_h 
        self.hidden_raw_x.append(hidden)
        squeezed = self.info_squeezing(hidden,activation)
        output = squeezed @ self.w_y + self.b_y
        h_output = squeezed 
        return h_output, output



    def forward(self, inputs, activation = 'tanh'):
        self.activation = activation
        one_hot_matrix = np.zeros((len(inputs), self.n_vocab), dtype=int)
        one_hot_matrix[np.arange(len(inputs)), inputs] = 1

        self.One_Hot = one_hot_matrix
        self.inputs = inputs

        self.hidden_raw_x = []
        self.input_x = []
        self.hidden_output_x = [np.zeros((1,self.hidden_size))]
        self.prob_output_x = []
        self.output_x = []

        x_p = np.zeros((1,self.hidden_size))

        for x_c in one_hot_matrix:
            x_c = x_c.reshape(1, -1)
            self.input_x.append(x_c)

            x_p, output = self.cell_forward(x_p,x_c,activation)

            self.output_x.append(output)
            self.hidden_output_x.append(x_p)
            sm = softmax()
            sm.forward(output)
            h_prob = sm.output
            self.prob_output_x.append(h_prob)
            



    def backward(self, dvalues_list):

        self.dw_x = np.zeros((self.n_vocab, self.hidden_size))
        self.dw_h = np.zeros((self.hidden_size, self.hidden_size))
        self.dw_y = np.zeros((self.hidden_size,self.n_vocab))

        self.db_h= np.zeros((1,self.hidden_size))
        self.db_y= np.zeros((1,self.n_vocab))

        dh_next = np.zeros((1,self.hidden_size))
        
        for time_stamp in range(len(self.inputs) - 1, -1, -1): 
            
            dvalues = dvalues_list[time_stamp]
            dvalues = dvalues.reshape(1, -1)

            self.dw_y += np.dot(self.hidden_output_x[time_stamp + 1].T, dvalues) 
            self.db_y += dvalues

            dvalues = np.dot(dvalues,self.w_y.T) + dh_next

            dvalues = self.info_unsqueeze(dvalues, self.hidden_output_x[time_stamp+1], self.activation)

            self.dw_h += np.dot(self.hidden_output_x[time_stamp].T, dvalues) 
            self.db_h += dvalues
            self.dw_x += np.dot(self.input_x[time_stamp].T, dvalues) 

            dh_next = np.dot(dvalues,self.w_h.T)
            self.d_inputs = dh_next



class RnnOptimizer:
    def __init__(self,lr = 0.001, beta = 0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self,layer):
        layer.w_x -= self.lr*layer.dw_x
        layer.w_h -= self.lr*layer.dw_h
        layer.w_y -= self.lr*layer.dw_y

        layer.b_h -= self.lr*layer.db_h
        layer.b_y -= self.lr*layer.db_y


    def OptimizerGD_WithMomentumn(self, layer):
        layer.w_x_moment = self.beta * layer.w_x_moment - self.lr*layer.dw_x
        layer.w_h_moment = self.beta * layer.w_h_moment - self.lr*layer.dw_h
        layer.w_y_moment = self.beta * layer.w_y_moment - self.lr*layer.dw_y

        layer.b_h_moment = self.beta * layer.b_h_moment - self.lr*layer.db_h
        layer.b_y_moment = self.beta * layer.b_y_moment - self.lr*layer.db_y


        layer.w_x += layer.w_x_moment
        layer.w_h += layer.w_h_moment
        layer.w_y += layer.w_y_moment

        layer.b_h += layer.b_h_moment
        layer.b_y += layer.b_y_moment
        


In [4]:
def OneHotMatrix(labels,num_classes):
    labels = np.asarray(labels, dtype=int).reshape(-1)
    y_true = np.zeros((len(labels), num_classes))
    y_true[np.arange(len(labels)), labels] = 1
    return y_true


def accuracy(y_pred, y_true):
    y_pred_labels = np.argmax(y_pred, axis=1)

    if len(y_true.shape) == 2:
        y_true_labels = np.argmax(y_true, axis=1)
    else:
        y_true_labels = y_true

    return np.mean(y_pred_labels == y_true_labels)

In [5]:
class Model:
    def __init__(self,layers):
        self.layers = layers

    def save(self,filename):
        with open (filename,'wb') as f:
            pickle.dump(self,f)

    @staticmethod
    def load(filename):
        with open(filename,'rb') as f:
            return pickle.load(f)
        
    def forward(self,X):
        output = X
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output

class Flatten:
    def forward(self, inputs):
        self.input_shape = inputs.shape
        self.output = inputs.reshape(inputs.shape[0], -1)

    def backward(self, dvalues):
        self.d_inputs = dvalues.reshape(self.input_shape)

data

In [6]:
data = "hello my name is evan and i'm in AI major ."
x = np.array(data.split())


le = LabelEncoder()
X = le.fit_transform(x).tolist()
mapping_dict = dict(zip(le.classes_, range(len(le.classes_))))

y = OneHotMatrix(X[1:],len(X))
    

In [7]:
print(X)
print(X[:-1])
print(X[1:])
print(len(X[1:])-1)

[4, 9, 10, 7, 3, 2, 5, 6, 1, 8, 0]
[4, 9, 10, 7, 3, 2, 5, 6, 1, 8]
[9, 10, 7, 3, 2, 5, 6, 1, 8, 0]
9


In [8]:
print(y)

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [9]:
#CONFIG
epoches = 1000
patience = 50
n_vocab = len(set(X))
hidden_size = 8
lowest_Loss = np.inf

In [10]:
rnn = Rnn(n_vocab,hidden_size)
opt = RnnOptimizer(lr=0.005)
activation = softmax()
L = Loss()
pat = 0


In [ ]:
for epoch in range(epoches):
    model = Model([rnn])
    rnn.forward(X[:-1])
    prob_list = rnn.prob_output_x
    loss_list = []
    dvalues_list = []
    total_loss = 0
    

    for t in range(len(X)-1):
        L.forward(prob_list[t],y[t:t+1] )
        loss = L.loss
        loss_list.append(loss)
        L.backward_with_softmax()
        dvalues = L.d_inputs
        dvalues_list.append(dvalues)
        total_loss += loss

    rnn.backward(dvalues_list)
    opt.OptimizerGD_WithMomentumn(rnn)

    print(f"epoch {epoch}, loss: {total_loss}")

    if total_loss <= lowest_Loss:
        lowest_Loss = total_loss
        model.save('best_model.pth')
        print("best model saved")
        pat = 0
    else:
        pat += 1

    if pat == patience:
        print("out of patience, train stopped")
        break

epoch 0, loss: 23.97906488251046
best model saved
epoch 1, loss: 23.974461765916203
best model saved
epoch 2, loss: 23.96579295978821
best model saved
epoch 3, loss: 23.9536029983152
best model saved
epoch 4, loss: 23.9384268678876
best model saved
epoch 5, loss: 23.92077688388934
best model saved
epoch 6, loss: 23.901132941957684
best model saved
epoch 7, loss: 23.8799358403998
best model saved
epoch 8, loss: 23.857583279993918
best model saved
epoch 9, loss: 23.834428104193137
best model saved
epoch 10, loss: 23.810778337054792
best model saved
epoch 11, loss: 23.786898598130676
best model saved
epoch 12, loss: 23.76301251388857
best model saved
epoch 13, loss: 23.739305796092783
best model saved
epoch 14, loss: 23.715929712615456
best model saved
epoch 15, loss: 23.693004730650653
best model saved
epoch 16, loss: 23.67062416302523
best model saved
epoch 17, loss: 23.64885769328074
best model saved
epoch 18, loss: 23.62775469349818
best model saved
epoch 19, loss: 23.607347280263344


In [22]:
def predict_next_word(prefix, rnn, le):
    if isinstance(prefix, str):
        prefix = prefix.split()

    unknown = [w for w in prefix if w not in le.classes_]
    if unknown:
        raise ValueError(f"Unknown words: {unknown}")

    ids = le.transform(prefix)

    rnn.forward(ids)
    res = []

    prob = rnn.prob_output_x
    for p in prob:
        pred_id = np.argmax(p, axis=1)[0]
        pri = le.inverse_transform([pred_id])[0]
        res.append(pri)

    return res

In [24]:
predict_next_word("hello", rnn, le)

[np.str_('my')]

In [ ]:
data = "hello my name is evan and i'm in AI major ."

In [15]:
d = data.split()

In [16]:
answer = []
for w in d:
    answer.append(np.array2string(predict_next_word(w, rnn, le)))

print(answer)

["'my'", "'name'", "'is'", "'evan'", "'and'", '"i\'m"', "'in'", "'AI'", "'major'", "'.'", "'evan'"]
